# Split2 and Join2

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from sympy import symbols

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler.registers.layout.component_partition import partition_signature
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register

## Split2

Split the packed left port `x` into the two right ports `y1` and
`y2`. Their widths add up to the input width. The first slice contains
the most significant bits. This operation preserves register data and
emits no gates.

In [ ]:
from flagquantum.compiler.registers.layout.partition import split_integer_pair, split_register_pair

### Example Instances

In [ ]:
n1, n2 = symbols("n1 n2")
ports = (Register("y1", opaque_register_type(n1)), Register("y2", opaque_register_type(n2)))
split = partition_signature(ports)
assert split[0].bitsize == n1 + n2

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Split2", split, {})])))

## Join2

Join the two left ports `y1` and `y2` into the packed right port `x`.
The parameter order and bit order are the same as Split2. Concrete
widths are required for wire or integer binding.

In [ ]:
from flagquantum.compiler.registers.layout.partition import join_integer_pair, join_register_pair

### Example Instances

In [ ]:
join = partition_signature(ports, reverse=True)
assert join[0].bitsize == n1 + n2

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Join2", join, {})])))

### Verify a concrete round trip

Bind a five-bit word with a two-bit prefix and a three-bit suffix,
then join the same physical wires. Integer values use the same
big-endian order.

In [ ]:
wires = (8, 3, 6, 4, 9)
first, second = split_register_pair(wires, 2, 3)
assert (first, second) == ((8, 3), (6, 4, 9))
assert join_register_pair(first, second) == wires
high, low = split_integer_pair(0b10110, 2, 3)
assert (high, low) == (0b10, 0b110)
assert join_integer_pair(high, low, 2, 3) == 0b10110